# Microarray SVA Demo

This notebook is a split-out module from the larger SVA demonstration project.

It includes:
- shared setup and helper functions
- a focused synthetic use case
- commented R code for teaching and adaptation


# Surrogate Variable Analysis (SVA) in Bioinformatics

This notebook is a **large, self-contained R notebook** that demonstrates:

- what surrogate variables are
- why hidden confounding matters in omics studies
- how SVA can be simulated and applied
- how it compares to naive analysis
- how it can be used in several synthetic but realistic bioinformatics use cases

Included use cases:
1. Bulk RNA-seq differential expression with hidden batch effects
2. Microarray-style gene expression with latent technical variation
3. DNA methylation probe analysis with hidden chip/position effects
4. Multi-cohort integration example
5. Case-control biomarker discovery with nuisance structure
6. Survival-associated expression signatures with latent confounding
7. eQTL-like setting with hidden sample structure
8. Pathway-level interpretation after SVA correction

The code is intentionally long and heavily commented so it can serve as a teaching demo.


## Notes

- The datasets are synthetic and made up for demonstration.
- The code is written in **R** and stored in notebook code cells.
- The notebook does not depend on any proprietary data.
- Some sections use package checks and fallbacks so the notebook remains readable even if not all packages are installed.


In [ ]:
# ============================================================
# SECTION 1: GLOBAL SETUP
# ============================================================

options(stringsAsFactors = FALSE)
set.seed(12345)

cat("Starting SVA demonstration notebook...\n")

required_pkgs <- c(
  "stats",
  "utils",
  "graphics",
  "grDevices",
  "methods"
)

optional_pkgs <- c(
  "sva",
  "limma",
  "ggplot2",
  "matrixStats",
  "survival"
)

check_package <- function(pkg) {
  is_installed <- requireNamespace(pkg, quietly = TRUE)
  message(sprintf("Package %-12s installed: %s", pkg, is_installed))
  return(is_installed)
}

pkg_status <- lapply(c(required_pkgs, optional_pkgs), check_package)
names(pkg_status) <- c(required_pkgs, optional_pkgs)

have_sva <- isTRUE(pkg_status[["sva"]])
have_limma <- isTRUE(pkg_status[["limma"]])
have_ggplot2 <- isTRUE(pkg_status[["ggplot2"]])
have_matrixStats <- isTRUE(pkg_status[["matrixStats"]])
have_survival <- isTRUE(pkg_status[["survival"]])

# Helper to safely library() optional packages if present
safe_library <- function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(
      library(pkg, character.only = TRUE)
    )
    return(TRUE)
  }
  FALSE
}

invisible(lapply(optional_pkgs, safe_library))

cat("Setup complete.\n")


In [ ]:
# ============================================================
# SECTION 2: HELPER FUNCTIONS
# ============================================================

scale_rows <- function(mat) {
  mat <- as.matrix(mat)
  row_means <- rowMeans(mat, na.rm = TRUE)
  row_sds <- apply(mat, 1, sd, na.rm = TRUE)
  row_sds[row_sds == 0] <- 1
  out <- (mat - row_means) / row_sds
  return(out)
}

simulate_hidden_factor <- function(n, k = 1, sd = 1) {
  matrix(rnorm(n * k, mean = 0, sd = sd), nrow = n, ncol = k)
}

simulate_feature_loadings <- function(p, k = 1, sd = 1) {
  matrix(rnorm(p * k, mean = 0, sd = sd), nrow = p, ncol = k)
}

make_design <- function(group) {
  model.matrix(~ group)
}

make_null_design <- function(n) {
  model.matrix(~ 1, data = data.frame(x = rep(1, n)))
}

pca_surrogates <- function(expr, n_sv = 2) {
  expr <- as.matrix(expr)
  expr_centered <- t(scale(t(expr), center = TRUE, scale = FALSE))
  pc <- prcomp(t(expr_centered), center = FALSE, scale. = FALSE)
  sv <- pc$x[, seq_len(min(n_sv, ncol(pc$x))), drop = FALSE]
  colnames(sv) <- paste0("PC_SV_", seq_len(ncol(sv)))
  return(sv)
}

estimate_surrogates <- function(expr, mod, mod0, n_sv = NULL) {
  expr <- as.matrix(expr)
  n <- ncol(expr)

  if (!is.null(n_sv) && n_sv < 1) {
    stop("n_sv must be >= 1 if provided")
  }

  if (have_sva) {
    if (is.null(n_sv)) {
      n_sv <- tryCatch({
        sva::num.sv(expr, mod, method = "leek")
      }, error = function(e) {
        2
      })
      if (is.na(n_sv) || n_sv < 1) n_sv <- 2
    }
    svobj <- tryCatch({
      sva::sva(expr, mod = mod, mod0 = mod0, n.sv = n_sv)
    }, error = function(e) {
      NULL
    })
    if (!is.null(svobj) && !is.null(svobj$sv)) {
      sv <- svobj$sv
      colnames(sv) <- paste0("SV", seq_len(ncol(sv)))
      return(list(method = "sva", sv = sv))
    }
  }

  sv <- pca_surrogates(expr, n_sv = ifelse(is.null(n_sv), 2, n_sv))
  return(list(method = "pca_fallback", sv = sv))
}

fit_featurewise_lm <- function(expr, design, coef_index = 2) {
  expr <- as.matrix(expr)
  XtX_inv <- solve(crossprod(design))
  betas <- XtX_inv %*% t(design) %*% t(expr)
  fitted <- design %*% betas
  resid <- t(expr) - fitted
  df_resid <- nrow(design) - ncol(design)
  sigma2 <- colSums(resid^2) / df_resid
  se_beta <- sqrt(sigma2 * XtX_inv[coef_index, coef_index])
  t_stat <- betas[coef_index, ] / se_beta
  p_val <- 2 * pt(abs(t_stat), df = df_resid, lower.tail = FALSE)

  out <- data.frame(
    beta = as.numeric(betas[coef_index, ]),
    t = as.numeric(t_stat),
    p = as.numeric(p_val),
    stringsAsFactors = FALSE
  )
  out$padj <- p.adjust(out$p, method = "BH")
  return(out)
}

evaluate_hits <- function(res, truth) {
  stopifnot(length(truth) == nrow(res))
  detected <- res$padj < 0.05
  tp <- sum(detected & truth)
  fp <- sum(detected & !truth)
  fn <- sum(!detected & truth)
  tn <- sum(!detected & !truth)

  sensitivity <- if ((tp + fn) == 0) NA_real_ else tp / (tp + fn)
  specificity <- if ((tn + fp) == 0) NA_real_ else tn / (tn + fp)
  precision <- if ((tp + fp) == 0) NA_real_ else tp / (tp + fp)
  fdr_empirical <- if ((tp + fp) == 0) NA_real_ else fp / (tp + fp)

  data.frame(
    TP = tp,
    FP = fp,
    FN = fn,
    TN = tn,
    sensitivity = sensitivity,
    specificity = specificity,
    precision = precision,
    empirical_FDR = fdr_empirical
  )
}

plot_pvalue_hist <- function(p, main = "P-value histogram") {
  hist(
    p,
    breaks = 40,
    main = main,
    xlab = "p-value",
    border = "white"
  )
}

plot_sv_vs_covariate <- function(sv, covariate, main_prefix = "Association") {
  sv <- as.matrix(sv)
  oldpar <- par(no.readonly = TRUE)
  on.exit(par(oldpar))
  par(mfrow = c(1, ncol(sv)))
  for (i in seq_len(ncol(sv))) {
    plot(
      covariate, sv[, i],
      xlab = deparse(substitute(covariate)),
      ylab = colnames(sv)[i],
      main = paste(main_prefix, colnames(sv)[i])
    )
    abline(lm(sv[, i] ~ covariate), lty = 2)
  }
}

confusion_summary <- function(name, eval_df) {
  cat("----", name, "----\n")
  print(eval_df)
  cat("\n")
}

top_table <- function(res, ids = NULL, n = 10) {
  tt <- res[order(res$p, decreasing = FALSE), , drop = FALSE]
  if (!is.null(ids)) {
    tt$id <- ids
    tt <- tt[, c("id", setdiff(colnames(tt), "id"))]
  }
  head(tt, n)
}

correlate_sv_with_known <- function(sv, meta_df) {
  sv <- as.matrix(sv)
  out <- list()
  for (j in seq_len(ncol(sv))) {
    vals <- sapply(meta_df, function(x) {
      if (is.numeric(x)) {
        suppressWarnings(cor(sv[, j], x))
      } else {
        x_num <- as.numeric(as.factor(x))
        suppressWarnings(cor(sv[, j], x_num))
      }
    })
    out[[colnames(sv)[j]]] <- vals
  }
  do.call(rbind, out)
}


In [ ]:
# ============================================================
# SECTION 4: USE CASE 2
# MICROARRAY-STYLE EXPRESSION DATA WITH LATENT TECHNICAL NOISE
# ============================================================

cat("Use case 2: Simulating microarray-like expression data...\n")

n_probes <- 6000
n_samples2 <- 72
phenotype <- factor(rep(c("A", "B"), times = c(36, 36)))

scanner_day <- factor(sample(paste0("day", 1:4), n_samples2, replace = TRUE))
operator <- factor(sample(c("op1", "op2", "op3"), n_samples2, replace = TRUE))

latent_temp <- scale(rnorm(n_samples2) + as.numeric(scanner_day))[, 1]
latent_humidity <- scale(rnorm(n_samples2) + as.numeric(operator) / 2)[, 1]

truth_probe <- rep(FALSE, n_probes)
truth_probe[sample(seq_len(n_probes), 500)] <- TRUE

baseline2 <- rnorm(n_probes, 7, 1.0)
pheno_effect <- rep(0, n_probes)
pheno_effect[truth_probe] <- rnorm(sum(truth_probe), 0.8, 0.25)

load_temp <- rnorm(n_probes, 0, 1.0)
load_hum <- rnorm(n_probes, 0, 0.8)

expr_micro <- matrix(0, nrow = n_probes, ncol = n_samples2)
for (i in seq_len(n_probes)) {
  expr_micro[i, ] <- baseline2[i] +
    pheno_effect[i] * (phenotype == "B") +
    load_temp[i] * latent_temp +
    load_hum[i] * latent_humidity +
    rnorm(n_samples2, 0, 0.8)
}

rownames(expr_micro) <- paste0("probe_", seq_len(n_probes))
colnames(expr_micro) <- paste0("array_", seq_len(n_samples2))

meta_micro <- data.frame(
  phenotype = phenotype,
  scanner_day = scanner_day,
  operator = operator,
  latent_temp = latent_temp,
  latent_humidity = latent_humidity
)

mod_micro <- model.matrix(~ phenotype + scanner_day + operator, data = meta_micro)
mod0_micro <- model.matrix(~ scanner_day + operator, data = meta_micro)

sv_micro <- estimate_surrogates(expr_micro, mod = mod_micro, mod0 = mod0_micro, n_sv = 4)
design_micro_sva <- cbind(mod_micro, sv_micro$sv)

res_micro_no_sva <- fit_featurewise_lm(expr_micro, mod_micro, coef_index = 2)
res_micro_sva <- fit_featurewise_lm(expr_micro, design_micro_sva, coef_index = 2)

eval_micro_no_sva <- evaluate_hits(res_micro_no_sva, truth_probe)
eval_micro_sva <- evaluate_hits(res_micro_sva, truth_probe)

confusion_summary("Microarray without SVA", eval_micro_no_sva)
confusion_summary("Microarray with SVA", eval_micro_sva)

cat("Correlations between estimated SVs and known metadata:\n")
print(round(correlate_sv_with_known(sv_micro$sv, meta_micro), 3))

oldpar <- par(no.readonly = TRUE)
par(mfrow = c(1, 2))
plot_pvalue_hist(res_micro_no_sva$p, main = "Microarray no SVA")
plot_pvalue_hist(res_micro_sva$p, main = "Microarray + SVA")
par(oldpar)
